In [ ]:
Google_Colab = True
if Google_Colab:
    !python -m pip install lightning
    !pip install nltk

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import torch
import os
if torch.cuda.is_available():
    device = torch.device("cuda")
    accelerator = "gpu"
    torch.cuda.memory.empty_cache()
    root_dir = "/content"
else:
    device = torch.device("cpu")
    accelerator = "cpu"
    root_dir = "."

project_name = "TextClassifier"
if not os.path.exists(f"{root_dir}/Checkpoints"):
    os.mkdir(f"{root_dir}/Checkpoints")
if not os.path.exists(f"{root_dir}/Checkpoints/{project_name}"):
    os.mkdir(f"{root_dir}/Checkpoints/{project_name}")


if not os.path.exists(f"{root_dir}/TensorBoard"):
    os.mkdir(f"{root_dir}/TensorBoard")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/train"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/train")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/validation"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/validation")

In [ ]:
Data = pd.read_csv(f"{root_dir}/ecommerceDataset.csv", header = None)
Data.head()

In [ ]:
Data.columns = ["Labels", "Data"]

# 1. Preprocessing

In [ ]:
Sentences = Data.iloc[:, 1]
Labels = Data.iloc[:, 0]

In [ ]:
# Label_Mapping
from sklearn.preprocessing import OneHotEncoder
one = OneHotEncoder()
one.fit(pd.DataFrame(Labels))
Labels_Encoded = one.transform(pd.DataFrame(Labels)).todense()
Label_Mapping = pd.concat((Labels, pd.DataFrame(Labels_Encoded.astype(int))), axis = 1)
Label_Mapping = Label_Mapping.drop_duplicates().reset_index(drop = True).set_index("Labels")
Label_Mapping

In [ ]:
Label_Mapping_dict = {int(np.argmax(Label_Mapping.loc[word, :])): word for word in Label_Mapping.index}
Label_Mapping_dict

In [ ]:
######################### Text Data preprocessing - Corpus for a Word2Vec
# from torchtext.data.utils import get_tokenizer
# tokenizer = get_tokenizer("basic_english")
from nltk.tokenize import word_tokenize as tokenizer
from tqdm import tqdm

#### Tokenize the Corpus
print("Tokenizing")
import nltk
nltk.download('punkt_tab')
Corpus = Sentences.to_numpy().tolist()
tokenized_Corpus = [tokenizer(str(sentence)) for sentence in Corpus]

### Remove Stopwords
print("Removing Stopwords")
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
tokenized_Corpus = [[token for token in Sentence if token.lower() not in stop_words] for Sentence in tokenized_Corpus]


# ### Stem tokens
print("Stemming")
from nltk.stem.snowball import EnglishStemmer
stemmer = EnglishStemmer()
stemmed_Corpus_nested =  [list(set([stemmer.stem(token) for token in Sentence])) for Sentence in tokenized_Corpus]
stemmed_Corpus_total = set(" ".join([" ".join(sentence) for sentence in stemmed_Corpus_nested]).split(" "))

del Corpus, tokenized_Corpus


################################# Creating a Word2Vec-embedding Model
# print("Training Word2Vec")
# from nltk.test.gensim_fixt import setup_module
# setup_module()
# import gensim
# Word2Vec_Model = gensim.models.Word2Vec(stemmed_Corpus_nested)
print("Done")

In [ ]:
!python --version

In [ ]:
### Preprocess the TextData for Training
Word_Mapping_dict = {token: i for i, token in enumerate(stemmed_Corpus_total)}
Vocab_size = len(Word_Mapping_dict)


def preprocess(Text):
    tokens = tokenizer(str(Text))
    tokens = [token for token in tokens if token.lower() not in stop_words]
    tokens = [stemmer.stem(token) for token in tokens]
    tokens = [Word_Mapping_dict[token] for token in tokens]
    return tokens

# I want to use as littel outisde models/function as possbile.
# I know that there a sentence embedding models.
# However, here I want to try it with word embedding
# this means, that all my Input sequencesa re of unqueal length.
# therefore, I pad the sequences beforehand so that they are all euqally long, hoping that the model ealrns to recognise the padding and to ignore it.
mapped_Sentences = Sentences.apply(lambda x: preprocess(x)).tolist()
longest_sentence = np.max([len(s) for s in mapped_Sentences])
mapped_padded_Sentences = [s + ([Vocab_size] * (longest_sentence - len(s))) for s in mapped_Sentences]

del stemmed_Corpus_nested, stemmed_Corpus_total, mapped_Sentences, Sentences

In [ ]:
#### Create Dataset and DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

class TextClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        sentence = np.array(self.X[idx])
        return torch.tensor(sentence, dtype = torch.long).squeeze(0), self.y[idx]

X_train, X_test, y_train, y_test = train_test_split(mapped_padded_Sentences, Labels_Encoded, test_size = 1/10)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 1/9)
# End Results: Train : Val : Test = 80 : 10 : 10
del mapped_padded_Sentences
Train_Set = TextClassificationDataset(X_train, y_train)
Val_Set = TextClassificationDataset(X_val, y_val)
Test_Set = TextClassificationDataset(X_test, y_test)


batch = 10
# workers = 11
Train_Loader = DataLoader(Train_Set, batch_size = batch, shuffle = True, pin_memory=True)
Val_Loader = DataLoader(Val_Set, batch_size = batch, shuffle = False, pin_memory=True)
Test_Loader = DataLoader(Test_Set, batch_size = batch, shuffle = False, pin_memory=True)

In [ ]:
from torch.amp import GradScaler, autocast
scaler = GradScaler("cuda")

from tqdm import tqdm
from torch import nn
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir = f"{root_dir}/TensorBoard/{project_name}")



class LearnedPositionalEmbedding(nn.Module):
    """
    From: https://medium.com/@benjybo7/unleash-the-power-of-positional-embeddings-5-techniques-and-how-to-implement-them-in-pytorch-8fc15d886c70
    """
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.position_embeddings = nn.Embedding(seq_len, d_model)

    def forward(self, input_ids):
        positions = torch.arange(0, input_ids.size(1), device=input_ids.device).unsqueeze(0)
        return self.position_embeddings(positions)

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedded_size, n_heads, depth, n_classes, project_name = "TextClassifier"):
        super().__init__()
        self.n_classes = n_classes
        self.InputEmbedding = nn.Embedding(vocab_size, embedded_size)
        self.PositionalEmbedding = LearnedPositionalEmbedding(vocab_size, embedded_size)
        self.TransformerEncoderLayer = nn.TransformerEncoderLayer(d_model = embedded_size, nhead = n_heads)
        self.TransformerEncoder = nn.TransformerEncoder(self.TransformerEncoderLayer, num_layers = depth)
        self.fc1 = nn.Linear(embedded_size, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, self.n_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()

        self.checkpoint_path = root_dir + "/Checkpoints/" + project_name

    def _base(self, x):
        x = self.InputEmbedding(x) + self.PositionalEmbedding(x)
        x = self.TransformerEncoder(x)
        x = self.relu(self.fc1(x))
        # x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        y_pred = self.fc4(x)
        y_pred = torch.sum(y_pred, dim = 1)
        return y_pred

    def training_step(self, batch):
        x, y = batch
        y_pred = self._base(x.to(device))
        loss = torch.nn.functional.cross_entropy(y_pred, y.squeeze(1).to(device))
        return loss

    def forward(self, x):
        y_pred = self._base(x)
        return y_pred


    def predict(self, x):
        y_pred = self._base(x.to(device))
        y_pred = torch.argmax(self.softmax(y_pred.resize(x.shape[0], self.n_classes)), dim = 1)
        y_pred_text = np.array([[Label_Mapping_dict[int(i)]] for i in y_pred])
        return y_pred_text

    def validation_step(self, batch):
        with torch.no_grad():
            loss = self.training_step(batch)
            return loss

    def save(self, epoch):
        name = f"TextClassifier_Epoch{epoch}.pt"
        torch.save(self.state_dict(), f"{self.checkpoint_path}/{name}")

    def load(self, name):
        self.load_state_dict(torch.load(f"{self.checkpoint_path}/{name}", weights_only=True))


epochs = 20
Model = TextClassifier(vocab_size = Vocab_size + 1, embedded_size = 100, n_heads = 5, depth = 1, n_classes = len(Label_Mapping_dict), project_name = project_name).to(device)
optimizer = torch.optim.Adam(Model.parameters(), lr=1e-4)
previous_checkpoints = os.listdir(Model.checkpoint_path)
gradient_accumulation = 10
if previous_checkpoints:
    initial_epoch = np.max([int(i.split("_Epoch")[-1].replace(".pt", "")) for i in previous_checkpoints if i.endswith(".pt")])
    Model.load(f"TextClassifier_Epoch{initial_epoch}.pt")
else:
    initial_epoch = 0

if ".earlystop" in previous_checkpoints or initial_epoch +1 == epochs:
    print("Model has already been fully trained.")
else:
    best_val_loss = np.inf
    earlyStopping_counter = 0
    earlyStopping_threshold = 5
    minimal_val_loss = 1e-10
    earlyStopping_procedure = False

    for epoch in tqdm(range(initial_epoch, epochs), desc = "Epochs:"):
        temp_train_loss = []
        temp_val_loss = []
        optimizer.zero_grad()
        optimizer_counter = 0
        for x, y in Train_Loader:
            optimizer_counter += 1
            with autocast("cuda"):
                loss = Model.training_step((x, y))
            scaler.scale(loss).backward() # scaler and autocast supposidly imporve memory usage by mixing 16 and 32 floating points
            if optimizer_counter % gradient_accumulation == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            temp_train_loss.append(loss.item())
            del loss
            ###### List to save train loss
        for x, y in Val_Loader:
            val_loss = Model.validation_step((x, y))
            temp_val_loss.append(val_loss.item())
            del val_loss
            ###### List to save val los
        median_train_loss = np.median(temp_train_loss)
        median_val_loss = np.median(temp_val_loss)
        writer.add_scalar("Loss/train", median_train_loss, epoch)
        writer.add_scalar("Loss/validation", median_val_loss, epoch)
        writer.flush()


        #### Early Stopping
        if median_val_loss + minimal_val_loss < best_val_loss: # current loss is smaller than the previous beats by at least the minimal value
            best_val_loss = median_val_loss
            earlyStopping_counter = 0
        else:
            earlyStopping_counter += 1

        if earlyStopping_counter >= earlyStopping_threshold: # too many epochs without improvement, initiate earlyStopping
            earlyStopping_procedure = True
        elif earlyStopping_counter == 0: # current val_loss is new best, make a savepoint
            Model.save(epoch = epoch)


        if earlyStopping_procedure:
            Path(f"{Model.checkpoint_path}/.earlystop").touch() #savefile so I now that earlyStopping has been performed and training is finished despite the maximum epochs not being reached
            #### EarlyStopping
            break

writer.close()
Path(f"{Model.checkpoint_path}/.earlystop").touch()

In [ ]:
### What is the accuarcy like if the model were to predict at random
possible_classes = list(Label_Mapping.index)
Random_Prediction_results = []
for Label in Labels:
    Choice = np.random.choice(possible_classes)
    acc = Label == Choice
    Random_Prediction_results.append(acc)
np.mean(Random_Prediction_results)

In [ ]:
test_loss_list = []
for x, y in Test_Loader:
    test_loss = Model.validation_step((x, y))
    test_loss_list.append(test_loss.item())
np.median(test_loss_list)

In [ ]:
test_acc_list = []
for x, y in Test_Loader:
    y_Pred = Model.predict(x)
    y_True = torch.argmax(nn.Softmax()(y.resize(x.shape[0], len(Label_Mapping.index))), dim = 1)
    y_True = np.array([[Label_Mapping_dict[int(i)]] for i in y_True])
    test_acc = np.mean(np.logical_and(y_Pred.squeeze(), y_True.squeeze()))
    test_acc_list.append(test_acc)
np.median(test_acc_list)